# Metro-ASR — Quick Start

**Egyptian Arabic + Code-Switching speech recognition**, non-autoregressive CTC on a
61.6M-parameter Conformer, with a detachable KenLM language head.

This notebook:
1. Installs Metro-ASR
2. Downloads a short public-domain Arabic clip to transcribe (or upload your own)
3. Transcribes with 3 lines of code
4. Adds beam search + a language model
5. Runs batch transcription and inspects timing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammedAly22/metro-asr/blob/main/examples/quick_start.ipynb)


## 1. Install

In [ ]:
import numpy
_COLAB_NUMPY = numpy.__version__
print(f"Colab's pre-installed numpy: {_COLAB_NUMPY} (pinning back to this after install)")


In [ ]:
!pip install -q "metro-asr[lm]"
!pip install -q "numpy=={_COLAB_NUMPY}"   # see note below


> [!NOTE]
> `pyctcdecode` (needed for beam search) still declares `numpy<2.0.0` in its own
> package metadata, even though it runs fine under numpy 2.x — pip will honor that and
> downgrade numpy to satisfy it. On Colab, which has numpy 2.x pre-installed and several
> other packages (jax, opencv, scipy, **pyarrow** — which `datasets` depends on) already
> built against that exact version, that downgrade breaks those packages' compiled
> extensions with errors like `numpy.dtype size changed` or `cannot import name
> '_center' from 'numpy._core.umath'` the next time anything imports them. Reinstalling
> *some* numpy 2.x isn't enough to fix this — it has to be the **exact** version Colab's
> other packages were compiled against, not just the latest release, so the cell above
> captures Colab's original version *before* installing anything and pins back to that
> precise version afterward.
>
> If you already hit a numpy-related crash before adding this fix, restart the
> runtime first (*Runtime → Restart session*) — otherwise the capture cell records
> the already-broken version instead of Colab's original one.

## 2. Get an audio clip

Upload your own `.wav`, or download one of this repo's demo clips (Egyptian Arabic,
real YouTube speech, used throughout the [model card](https://huggingface.co/MohammedAly22/Metro-ASR-Small)
and the [evaluation report](https://mohammedaly22.github.io/metro-asr/)).

In [ ]:
!wget -q -O audio.wav https://raw.githubusercontent.com/MohammedAly22/metro-asr/main/test_samples/5.wav
print("Downloaded audio.wav")


## 3. Transcribe (3 lines)

In [ ]:
from metro_asr import MetroASREngine

# Auto-downloads the acoustic model + tokenizer from HuggingFace on first use,
# caches under ~/.cache/metro-asr afterwards.
engine = MetroASREngine.from_pretrained("small")

result = engine.transcribe("audio.wav")
print(result.text)


## 4. Beam search + language model

`lm_path="auto"` fetches the released KenLM 5-gram (~6 GB) alongside the acoustic model —
only needed once, and only if you want beam search. Greedy decoding above never downloads it.

In [ ]:
engine = MetroASREngine.from_pretrained(
    "small",
    lm_path="auto",     # fetches lm_5gram.bin from the HF repo; ~6 GB, one-time download
    beam_width=100,
    lm_alpha=0.5,
    lm_beta=5.0,
)

greedy = engine.transcribe("audio.wav")
beam = engine.transcribe("audio.wav", beam_search=True)

print(f"Greedy:  {greedy.text}")
print(f"Beam+LM: {beam.text}")


## 5. Batch transcription

Reuses the engine already loaded above — no need to re-download anything.

In [ ]:
audio_files = ["audio.wav"]  # add more paths here
results = engine.transcribe_batch(audio_files)

for path, result in zip(audio_files, results):
    print(f"{path}: {result.text}")
    print(f"  duration={result.duration:.1f}s  rtf={result.rtf:.4f}  "
          f"({1/result.rtf:.0f}x real time)")


## 6. Every field on the result

In [ ]:
result = engine.transcribe("audio.wav")

print(f"Text:         {result.text}")
print(f"Duration:     {result.duration:.2f}s")
print(f"Inference:    {result.inference_time*1000:.1f}ms")
print(f"Decoding:     {result.decoding_time*1000:.1f}ms")
print(f"RTF:          {result.rtf:.6f}  ({1/result.rtf:.0f}x real time)")
print(f"Method:       {result.method}")
print(f"Params:       {engine.param_count:,}")


## Next steps

- **[Fine-tuning](fine_tuning.ipynb)** — adapt Metro-ASR to your own domain or dataset
- **[Serving & streaming](streaming_server.ipynb)** — REST API and real-time transcription
- **[Full README](https://github.com/MohammedAly22/metro-asr#readme)** — architecture, training
  from scratch, building your own domain language head
- **[Evaluation report](https://mohammedaly22.github.io/metro-asr/)** — measured greedy vs.
  beam+LM accuracy, and general vs. domain-specialised language heads, on real audio
